In [6]:
!pip install "accelerate>=1.1.0" "transformers[torch]" datasets

In [7]:
import pandas as pd
import torch
import os
from transformers import BartTokenizer, BartForConditionalGeneration, Seq2SeqTrainer, Seq2SeqTrainingArguments
from datasets import Dataset

# Assuming the notebook is inside the /notebooks folder
data_path = "/kaggle/input/datasets/subhan1501/text-summarizatio/arxiv_sample.csv"

print("Loading massive dataset...")
# We use nrows=75000 to safely stay under Kaggle's 12-hour timeout limit!
df = pd.read_csv(data_path, nrows=75000) 

# Drop any rows with missing data just to be safe
df = df.dropna(subset=['abstract', 'title'])
print(f"Loaded {len(df)} clean rows for training.")

Loading massive dataset...
Loaded 75000 clean rows for training.


In [8]:
model_name = "facebook/bart-large-cnn"
print(f"Downloading {model_name}...")

tokenizer = BartTokenizer.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name)

dataset = Dataset.from_pandas(df)

def tokenize_function(examples):
    model_inputs = tokenizer(examples["abstract"], max_length=128, truncation=True, padding="max_length")
    labels = tokenizer(examples["title"], max_length=32, truncation=True, padding="max_length")
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print("Tokenizing dataset (this might take a few minutes)...")
tokenized_datasets = dataset.map(tokenize_function, batched=True)

Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

Tokenizing dataset (this might take a few minutes)...


Map:   0%|          | 0/75000 [00:00<?, ? examples/s]

In [9]:
# Create the output directory in your models folder
save_path = "../models/Model_BART"
os.makedirs(f"{save_path}/checkpoints", exist_ok=True)

print(f"Hardware Check: Using {'GPU' if torch.cuda.is_available() else 'CPU'}")

args = Seq2SeqTrainingArguments(
    output_dir=f"{save_path}/checkpoints",
    learning_rate=2e-5,
    per_device_train_batch_size=8, 
    weight_decay=0.01,
    
    # 💥 CRITICAL FIXES BELOW 💥
    save_strategy="epoch", # Only save at the very end of the epoch, preventing disk crashes!
    save_total_limit=1,    # Only keep 1 file on the Kaggle hard drive
    num_train_epochs=1,    # Reduced to 1 so it finishes in a few hours
    
    logging_steps=100,
    eval_strategy="no",
    fp16=True 
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets,
    processing_class=tokenizer,
)

print("🔥 Starting GPU Training Loop...")
trainer.train()

# Save the final fine-tuned model
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"✅ Training complete! Model securely saved to {save_path}")

Hardware Check: Using GPU
🔥 Starting GPU Training Loop...


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss
100,4.250067
200,2.707551
300,2.648698
400,2.574151
500,2.495338
600,2.495360
700,2.472151
800,2.465527
900,2.358857
1000,2.406484


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Training complete! Model securely saved to ../models/Model_BART


In [10]:
import shutil
import os
from IPython.display import FileLink

# The path where we saved the model in Cell 4
model_folder = "../models/Model_BART"
zip_filename = "BART_Finetuned_Model" # Do not add .zip here, shutil handles it

print("📦 Zipping the fine-tuned model (this may take a minute)...")
shutil.make_archive(zip_filename, 'zip', model_folder)
print("✅ Zip created successfully!")

print("Click the link below to download manually if the auto-download fails:")
display(FileLink(f"{zip_filename}.zip"))

📦 Zipping the fine-tuned model (this may take a minute)...
✅ Zip created successfully!
Click the link below to download manually if the auto-download fails:


/kaggle/working/BART_Finetuned_Model.zip

In [11]:
from IPython.display import HTML

# HTML and Javascript to automatically click the download link
auto_download_code = f"""
<a id="auto_download" href="{zip_filename}.zip" download="{zip_filename}.zip" style="display:none;"></a>
<script>
    setTimeout(function() {{
        document.getElementById('auto_download').click();
    }}, 1000); // 1 second delay to ensure the file is ready
</script>
"""

display(HTML(auto_download_code))
print("⬇️ Automatic download triggered!")
print("⚠️ Note: If nothing happens, check your browser's address bar—it might be blocking pop-ups/downloads from Kaggle.")

⬇️ Automatic download triggered!
⚠️ Note: If nothing happens, check your browser's address bar—it might be blocking pop-ups/downloads from Kaggle.
